# MovieLens 100K: Exploratory Data Analysis

This notebook explores the MovieLens 100K dataset to understand its structure, rating distributions, user and movie activity, and data quality before implementing recommendation models.

## 1. Dataset Overview

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import GENRE_COLS, load_movielens

sns.set_theme(style="whitegrid")

### Load Dataset

In [ ]:
ratings, movies, users = load_movielens()

### Verify Dataset Dimensions

In [ ]:
for name, df in zip(["Ratings", "Movies", "Users"], [ratings, movies, users]):
    print(f"{name}: {df.shape}")

### Examine Unique Users and Movies

In [ ]:
# We wish to determine the number of distinct users and movies in the ratings data.
print("Unique users:", ratings["user_id"].nunique())
print("Unique movies:", ratings["movie_id"].nunique())
print("Total ratings:", len(ratings))

## 2. Rating Distribution

### Rating Frequencies

In [ ]:
freq = ratings["rating"].value_counts().sort_index()
percentage = (freq / len(ratings)) * 100

rating_distribution = pd.DataFrame({
    "Frequency": freq,
    "Percentage (%)": percentage
})

display(rating_distribution)

### Descriptive Statistics

In [ ]:
display(ratings["rating"].describe())

print("Median rating:", ratings["rating"].median())
print("Mode rating:", ratings["rating"].mode().tolist())

### Visualize Rating Distribution

In [ ]:
# Visualization of Rating Distribution
# This shows how frequently users assign each rating
plt.figure(figsize=(8, 5))
sns.barplot(x=freq.index, y=freq.values)

plt.title("Distribution of Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.xticks(range(5), [1, 2, 3, 4, 5])

plt.show()

## 3. Movie Activity Analysis

### Ratings per Movie

In [ ]:
movie_activity = ratings.groupby("movie_id").size()

display(movie_activity.describe())

### Most Frequently Rated Movies

In [ ]:
top_movies = (
    movie_activity
    .nlargest(10)
    .rename("rating_count")
    .reset_index()
    .merge(
        movies[["movie_id", "title"]],
        on="movie_id",
        how="left"
    )
    .sort_values("rating_count", ascending=False)
)

display(top_movies[["movie_id", "title", "rating_count"]])

### Movie Activity Distribution

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(movie_activity, bins=30)

plt.title("Distribution of Ratings per Movie")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Movies")

plt.tight_layout()
plt.show()

## 4. User Activity Analysis

### Ratings per User

In [ ]:
user_activity = ratings.groupby("user_id").size()

display(user_activity.describe())

### Most Active Users

In [ ]:
top_users = (
    user_activity
    .nlargest(10)
    .rename("rating_count")
    .reset_index()
)

display(top_users)

### User Activity Distribution

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(user_activity, bins=30)

plt.title("Distribution of Ratings per User")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Users")

plt.tight_layout()
plt.show()

## 5. Data Quality Assessment

### Missing Values

In [ ]:
print("Missing values in ratings:")
display(ratings.isnull().sum())

print("Missing values in movies:")
display(movies.isnull().sum())

print("Missing values in users:")
display(users.isnull().sum())

### Duplicate Records

In [ ]:
print("Duplicate rating records:", ratings.duplicated().sum())
print("Duplicate movie records:", movies.duplicated().sum())
print("Duplicate user records:", users.duplicated().sum())

print(
    "Duplicate user-movie pairs:",
    ratings.duplicated(subset=["user_id", "movie_id"]).sum()
)

dup_titles = movies[movies["title"].duplicated(keep=False)].sort_values("title")
print("Rows sharing a duplicate movie title:", len(dup_titles))
print("Distinct movie titles appearing multiple times:", dup_titles["title"].nunique())
display(dup_titles[["movie_id", "title", "release_date"]].head(10))

### Invalid Ratings

In [ ]:
invalid_ratings = ratings[
    ~ratings["rating"].between(1, 5)
]

print("Invalid ratings:", len(invalid_ratings))
display(invalid_ratings.head())

### Dataset Consistency

In [ ]:
missing_users = set(ratings["user_id"]) - set(users["user_id"])
missing_movies = set(ratings["movie_id"]) - set(movies["movie_id"])

print("User IDs without metadata:", len(missing_users))
print("Movie IDs without metadata:", len(missing_movies))

## 6. Genre, Sparsity, and Modeling Readiness

### Interaction Matrix Sparsity and Temporal Span

In [ ]:
n_users = ratings["user_id"].nunique()
n_movies = ratings["movie_id"].nunique()
sparsity = 1.0 - len(ratings) / (n_users * n_movies)
print(f"User-movie matrix size: {n_users} x {n_movies} = {n_users * n_movies:,} cells")
print(f"Observed ratings:       {len(ratings):,} ({100 * (1 - sparsity):.2f}% filled)")
print(f"Matrix sparsity:        {100 * sparsity:.2f}%")

timestamps = pd.to_datetime(ratings["timestamp"], unit="s")
span_days = (timestamps.max() - timestamps.min()).days
print(f"Earliest rating: {timestamps.min()}")
print(f"Latest rating:   {timestamps.max()}")
print(f"Collection span: {span_days} days")

monthly_counts = timestamps.dt.to_period("M").value_counts().sort_index()
plt.figure(figsize=(9, 4))
monthly_counts.plot(kind="bar", color="steelblue")
plt.title("Monthly Rating Volume Over Collection Period")
plt.xlabel("Month")
plt.ylabel("Number of Ratings")
plt.tight_layout()
plt.show()

### Positive Rating Split (Threshold >= 4)

In [ ]:
is_positive = ratings["rating"] >= 4
pos_share = is_positive.mean() * 100
print(f"Positive ratings (>= 4): {is_positive.sum():,} ({pos_share:.2f}%)")
print(f"Negative/neutral (< 4):  {(~is_positive).sum():,} ({100 - pos_share:.2f}%)")

user_positives = (
    ratings.assign(positive=is_positive)
    .groupby("user_id")["positive"]
    .sum()
)
print("\nPositive ratings per user:")
display(user_positives.describe())
print("Users with 0 positive ratings:", int((user_positives == 0).sum()))

### Genre Distribution and Binary Feature Verification

In [ ]:
genre_values = movies[GENRE_COLS]
unique_vals = sorted(pd.unique(genre_values.to_numpy().ravel()).tolist())
genres_per_movie = genre_values.sum(axis=1)

print("Unique genre indicator values:", unique_vals)
print("Movies with zero genre flags:", int((genres_per_movie == 0).sum()))
unknown_movies = movies[movies["unknown"] == 1][["movie_id", "title"]]
print("Movies tagged with 'unknown' genre:", len(unknown_movies))
display(unknown_movies)

print("\nGenres per movie:")
display(genres_per_movie.describe())

genre_counts = genre_values.sum().sort_values(ascending=True)
plt.figure(figsize=(9, 6))
genre_counts.plot(kind="barh", color="teal")
plt.title("Number of Movies per Genre")
plt.xlabel("Number of Movies")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

display(genre_counts.sort_values(ascending=False).rename("movie_count").to_frame())

### Long-Tail Coverage vs. Minimum Ratings Threshold

In [ ]:
thresholds = [1, 10, 25, 50, 100, 200]
total_catalog = len(movies)
total_ratings = len(ratings)

coverage_rows = []
for min_r in thresholds:
    eligible = movie_activity[movie_activity >= min_r]
    coverage_rows.append({
        "min_ratings": min_r,
        "eligible_movies": len(eligible),
        "catalog_share_%": 100 * len(eligible) / total_catalog,
        "ratings_covered_%": 100 * eligible.sum() / total_ratings,
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.round(2))

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.bar(coverage_df["min_ratings"].astype(str), coverage_df["eligible_movies"], color="slateblue", alpha=0.8)
ax1.set_xlabel("min_ratings threshold")
ax1.set_ylabel("Eligible Movies (count)", color="slateblue")
ax2 = ax1.twinx()
ax2.plot(coverage_df["min_ratings"].astype(str), coverage_df["catalog_share_%"], color="darkorange", marker="o", linewidth=2)
ax2.set_ylabel("Catalog Share (%)", color="darkorange")
plt.title("Eligible Popularity Catalog vs. min_ratings Threshold")
plt.tight_layout()
plt.show()

## 7. Key Findings

### Dataset Overview

[Document the verified number of users, movies, and ratings.]

### Rating Distribution

[Identify the most frequent rating, average rating, and general rating tendencies.]

### Movie Activity

[Identify the most frequently rated movies and describe the distribution of ratings across movies.]

### User Activity

[Describe the distribution of user rating activity and identify any notable patterns.]

### Data Quality

[Summarize missing values, duplicate records, invalid ratings, and inconsistencies.]

### Conclusion

[Summarize the most important observations from the exploratory analysis.]